# Résoudre `7_big` avec OR-Tools
Dernier jeu de données : **50 000 modèles et 250 000 datasets**. *« We're in the endgame now »*. Pas de nouvelle idée de modélisation : `7_big` ressemble à une version agrandie de `2_medium`. Le défi est la **taille**.

Ce notebook utilise directement `pipeline.run`, l'enchaînement complet que lance aussi `make solve` : contrôle préalable, solveur, attribution selon plusieurs ordres, complétion, réparation.

**Version 3.** Ce notebook n'implémente plus rien lui-même : il appelle le package `ia_responsable` (dossier `src/`). Le récit et les expériences sont ceux de la version précédente ; le code est celui que teste `make test` et qu'utilise `make solve`. Les fonctions sont commentées dans les modules indiqués entre parenthèses.

**Durée d'exécution** : environ 30 minutes (dont 4 pour le contrôle préalable et 10 pour le solveur).

## 1. Charger l'instance

In [1]:
from pathlib import Path
import json, time, random
from collections import Counter
import pandas as pd

from ia_responsable import load
from ia_responsable.cli import official_score
from ia_responsable.fill import fill, give, new_pools, no_full_models
from ia_responsable.model import solve
from ia_responsable.packing import REFERENCE_ORDERS, noisy_orders, pack
from ia_responsable.pipeline import plan_value, run
from ia_responsable.repair import State, complete, repair

pd.set_option("display.width", 150)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
inst = load(ROOT / "datasets" / "7_big.json")
WINNER = 5_822_561


def score(plan):
    """Score officiel (test_solution.py). Lève une erreur si le plan est invalide."""
    s, valid, message = official_score(ROOT, inst.text, plan)
    assert valid, message
    return s


def solver_row(info, sel):
    return dict(statut=info.status, objectif=info.objective, borne=round(info.bound, 1), écart_prouvé=round(info.gap, 1),
                secondes=round(info.seconds), pleins=len(sel.full), demi=len(sel.half), sources=len(sel.sources))


def failed_full(sel, plan):
    """Modèles choisis à pleine valeur qui n'ont pas eu leurs données libres."""
    data = {}
    for d, m in plan["dataMappings"]:
        data.setdefault(m, []).append(d)
    return [m for m in sel.full if m not in data or any(inst.copyrighted[d] for d in data[m])]


def save_if_better(plan):
    out = ROOT / "solutions" / f"{inst.name}_main.json"
    new = score(plan)
    current = official_score(ROOT, inst.text, json.loads(out.read_text()))[0] if out.exists() else 0
    if new > current:
        out.write_text(json.dumps(plan))
        print(f"{out.name} mis à jour : {current:,} -> {new:,}")
    else:
        print(f"{out.name} inchangé ({current:,}) ; ce notebook a obtenu {new:,}")


print(f"{inst.name} : {len(inst.value)} modèles, {len(inst.size)} datasets | gagnant du concours : {WINNER:,}")

7_big : 50000 modèles, 250000 datasets | gagnant du concours : 5,822,561


## 2. Ce qui change

In [2]:
share = inst.energy_cap / sum(inst.energy.values())
demand = {t: sum(low.get(t, 0) for low in inst.lower.values()) for t in "ntic"}
print(f"énergie de tous les modèles : {1 / share:.2f} × le plafond | sources possibles : {len(inst.mono)}")
print("instance.data_is_scarce :", inst.data_is_scarce)
pd.DataFrame({"offre_libre": inst.free_supply, "demande_probable": {t: round(demand[t] * share) for t in "ntic"},
              "demande_probable_sur_offre_libre": {t: round(demand[t] * share / inst.free_supply[t], 2) for t in "ntic"}})

énergie de tous les modèles : 5.82 × le plafond | sources possibles : 21094
instance.data_is_scarce : False


,offre_libre,demande_probable,demande_probable_sur_offre_libre
n,105406037,101228920,0.96
t,105427627,102015459,0.97
i,105818003,100837797,0.95
c,106153061,100267925,0.94


**Lecture**
- L'énergie limite : on entraînera environ un modèle sur six.
- Les modèles que l'énergie permet demanderaient à peu près autant de données que l'offre **libre** : elles seront juste suffisantes, comme sur `2_medium`.
- Les données **totales** ne manquent pas (`data_is_scarce` est faux) : le package n'autorise donc les sources que pour les modèles impossibles.

## 3. Le meilleur modèle est celui qui suffit
Pendant la préparation, on a lancé les deux versions du modèle 10 minutes chacune :

| Version | Solution après 10 min | Borne |
|---|---|---|
| sources pour les besoins impossibles seulement | 5 823 205 | 5 823 268 |
| sources partout (environ 100 000 variables de plus) | 4 348 180 | 37 340 527 |

Le gros modèle n'a même pas dépassé les premières étapes de la recherche. C'est le contraire de `5_energy` et `6_shortage`, où les sources étaient indispensables.

## 4. Le pipeline complet (`pipeline.py`)
Adaptations à la taille, déjà intégrées dans les réglages de `cli.py` :
- une attribution prend plus d'une minute (250 000 datasets) : seulement les deux ordres de référence. Des ordres bruités ont été essayés : ils faisaient moins bien ;
- la réparation est limitée à 20 partenaires d'échange.

Une piste essayée sans succès : une marge de 0,2 % sur les données libres. Plus aucun modèle n'échoue, mais la sélection perd environ 4 200 points. Ici, chaque dixième de pourcent de données libres vaut environ 2 000 points : mieux vaut les utiliser toutes et réparer.

In [3]:
t0 = time.time()
plan, report = run(inst, seconds=600)
print(f"{time.time() - t0:.0f} s")
info = report.solver
print(f"SCIP {info.status} : objectif {info.objective:,}, borne {info.bound:,.1f}")
pd.DataFrame(report.orders).set_index("ordre")

1734 s
SCIP FEASIBLE : objectif 5,823,205, borne 5,823,267.7


,ratés,après_attribution,ajouts,réparations,score
ordre,,,,,
petits besoins,13,5821868,6,13,5822283
plus contraints,4,5821238,6,10,5822383


In [4]:
s = score(plan)
print(f"score officiel : {s:,} | écart au gagnant : {s - WINNER:+,} ({s / WINNER:.4%})")
save_if_better(plan)

score officiel : 5,822,383 | écart au gagnant : -178 (99.9969%)


7_big_main.json inchangé (5,822,383) ; ce notebook a obtenu 5,822,383


## 5. Bilan des sept jeux de données
Au concours, chaque jeu de données rapporte `1 000 000 × score / meilleur score`. On relit ici les solutions de `solutions/` et on les fait noter par le correcteur officiel.

*Note : ce bilan a été calculé juste avant la nouvelle exécution du notebook 11, qui a porté `2_medium` de 272 224 à 272 271. Réexécutez la cellule ci-dessous pour obtenir le bilan à jour (total : 6 998 113 points).*

In [5]:
WINNERS = {"1_example": 12_946, "2_medium": 272_322, "3_free": 523_758, "4_precise": 1_031_941,
           "5_energy": 3_786_108, "6_shortage": 669_798, "7_big": 5_822_561}
rows = []
for name, best in WINNERS.items():
    data = ROOT / "datasets" / f"{name}.json"
    sol = ROOT / "solutions" / f"{name}_main.json"
    if not (data.exists() and sol.exists()):
        continue
    ours, valid, _ = official_score(ROOT, data.read_text(), json.loads(sol.read_text()))
    rows.append(dict(dataset=name, notre_score=ours, gagnant=best, écart=ours - best,
                     points_classement=round(1_000_000 * ours / best), valide=valid))
recap = pd.DataFrame(rows).set_index("dataset")
display(recap)
print(f"total : {recap.points_classement.sum():,} points sur {1_000_000 * len(recap):,} ({recap.points_classement.sum() / (1_000_000 * len(recap)):.3%})")

,notre_score,gagnant,écart,points_classement,valide
dataset,,,,,
1_example,12946,12946,0,1000000,True
2_medium,272224,272322,-98,999640,True
3_free,523758,523758,0,1000000,True
4_precise,1031910,1031941,-31,999970,True
5_energy,3785420,3786108,-688,999818,True
6_shortage,668822,669798,-976,998543,True
7_big,5822383,5822561,-178,999969,True


total : 6,997,940 points sur 7,000,000 (99.971%)


## À retenir (sur l'ensemble du parcours)
1. **Regarder les données avant de choisir l'algorithme** : chaque instance avait sa contrainte active.
2. **L'exact et l'heuristique se complètent** : le solveur choisit, les gloutons attribuent, la réparation corrige.
3. **Une borne est un instrument de mesure** : elle a révélé une hypothèse fausse (notebook 10) et un faux optimum numérique (notebook 14).
4. **Un solveur se règle et peut se tromper** : tolérance d'arrêt, contrainte oubliée, coefficients trop grands, limite de temps.
5. **À grande échelle, on allège** : le meilleur pipeline est celui qu'on peut exécuter.
6. **Les tests trouvent ce que les notebooks cachent** : en passant au package, un test a révélé qu'un modèle à demi-valeur ne pouvait pas mélanger datasets libres et sous droits dans un même besoin (`fill.fill_mixed`).